# 02 - Data Preparation

## Objective

This notebook converts the transaction data into a daily product time series.

| Item | Definition |
| --- | --- |
| Target | `quantity_sold` |
| Forecast horizon | 14 days |
| Granularity | 1 row = 1 product per day |

**Input:** `data/raw/motoretail.csv`  
**Output:** `data/processed/daily_product_sales.csv`  
**Next notebook:** `03_baseline_forecast.ipynb`

## Imports and Load Dataset

In [1]:
import os
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_PATH = ROOT / "data" / "raw" / "motoretail.csv"
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (6875, 27)


,sale_date,sale_time,day_of_week,month,customer_type,customer_city,customer_state,customer_neighborhood,customer_address,customer_zipcode,...,estimated_profit_brl,payment_method,sales_channel,weather_condition,delivery_app_peak,is_weekend,is_holiday_campaign,current_stock_snapshot,supplier_lead_time_days,returned
0,2025-01-01,16:00,Quarta-feira,Janeiro,Novo,Caldas Novas,GO,Itanhangá II,"Rua São Paulo, 530",75680-195,...,153.75,Pix,Loja Física,Ensolarado,True,False,False,102,10,False
1,2025-01-01,14:30,Quarta-feira,Janeiro,Recorrente,Caldas Novas,GO,Jardim Belvedere,"Rua do Balneário, 3537",75685-204,...,104.78,Pix,Instagram,Ensolarado,False,False,False,23,9,False
2,2025-01-01,13:15,Quarta-feira,Janeiro,Recorrente,Caldas Novas,GO,Portal das Águas,"Av. Orcalino Santos, 760",75683-891,...,84.85,Pix,Loja Física,Ensolarado,True,False,False,54,9,False
3,2025-01-01,18:00,Quarta-feira,Janeiro,Novo,Caldas Novas,GO,Itanhangá II,"Rua do Balneário, 8761",75683-267,...,42.55,Pix,Instagram,Ensolarado,False,False,False,47,7,False
4,2025-01-01,11:30,Quarta-feira,Janeiro,Novo,Caldas Novas,GO,Nova Vila,"Rua São Paulo, 6492",75687-246,...,144.42,Pix,WhatsApp,Ensolarado,True,False,False,93,8,False


## Convert Dates and Sort Transactions

`sale_time` is used only to identify the last stock and lead-time snapshots inside each product-day. It is not used as a model feature.

In [2]:
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="raise")
df["sale_timestamp"] = pd.to_datetime(
    df["sale_date"].dt.strftime("%Y-%m-%d") + " " + df["sale_time"],
    errors="raise",
)
df = df.sort_values(["product_name", "sale_timestamp"]).copy()

print("Date range:", df["sale_date"].min(), "to", df["sale_date"].max())
print("Products:", df["product_name"].nunique())

Date range: 2025-01-01 00:00:00 to 2026-05-29 00:00:00
Products: 12


## Create a Date-Level Weather Category

Some dates contain more than one weather label. Weather is treated as a category, and the most frequent label on each date is used. Ties use the first label in sorted order so the result is reproducible.

In [3]:
def most_frequent_value(values):
    return values.mode().sort_values().iloc[0]

weather_by_date = (
    df.groupby("sale_date")["weather_condition"]
    .agg(most_frequent_value)
    .rename("weather_condition")
    .reset_index()
)

assert weather_by_date["sale_date"].is_unique
weather_by_date.head()

,sale_date,weather_condition
0,2025-01-01,Ensolarado
1,2025-01-02,Ensolarado
2,2025-01-03,Ensolarado
3,2025-01-04,Ensolarado
4,2025-01-05,Ensolarado


## Aggregate Transactions by Product and Date

Sales, revenue and profit are flows, so they are summed. Price and discount use the daily mean. Stock and supplier lead time are snapshots, so the last transaction value of the day is used instead of a sum or mean.

In [4]:
daily_sales = (
    df.groupby(["product_name", "sale_date"], as_index=False, observed=True)
    .agg(
        quantity_sold=("quantity_sold", "sum"),
        total_revenue_brl=("total_revenue_brl", "sum"),
        estimated_profit_brl=("estimated_profit_brl", "sum"),
        discount_pct=("discount_pct", "mean"),
        unit_price_brl=("unit_price_brl", "mean"),
        current_stock_snapshot=("current_stock_snapshot", "last"),
        supplier_lead_time_days=("supplier_lead_time_days", "last"),
    )
)

product_category = (
    df.groupby("product_name")["product_category"]
    .agg(most_frequent_value)
    .rename("product_category")
    .reset_index()
)

print("Observed product-days:", len(daily_sales))
daily_sales.head()

Observed product-days: 3461


,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,supplier_lead_time_days
0,Bag Delivery 45L,2025-01-01,5,889.97,414.97,7.5,178.1725,101,3
1,Bag Delivery 45L,2025-01-02,2,364.80,174.80,0.0,182.4000,53,7
2,Bag Delivery 45L,2025-01-03,1,183.92,88.92,5.0,183.9200,24,7
3,Bag Delivery 45L,2025-01-04,3,536.25,251.25,5.0,178.7500,52,7
4,Bag Delivery 45L,2025-01-05,8,1442.82,682.82,4.0,179.0000,109,8


## Add Missing Product-Days

A complete product × date grid is needed because no transaction on a day means zero observed sales. Flow columns are filled with zero. Product category is filled from the product mapping. Price, stock and lead time are carried forward because they are last-known business values; initial rows before the first observation may remain missing and are reported below.

In [5]:
full_grid = pd.MultiIndex.from_product(
    [
        sorted(df["product_name"].unique()),
        pd.date_range(df["sale_date"].min(), df["sale_date"].max(), freq="D"),
    ],
    names=["product_name", "sale_date"],
).to_frame(index=False)

df_daily = (
    full_grid.merge(daily_sales, on=["product_name", "sale_date"], how="left")
    .merge(product_category, on="product_name", how="left")
    .merge(weather_by_date, on="sale_date", how="left")
    .sort_values(["product_name", "sale_date"])
    .reset_index(drop=True)
)

flow_columns = ["quantity_sold", "total_revenue_brl", "estimated_profit_brl"]
df_daily[flow_columns] = df_daily[flow_columns].fillna(0)
df_daily["discount_pct"] = df_daily["discount_pct"].fillna(0)

snapshot_columns = ["unit_price_brl", "current_stock_snapshot", "supplier_lead_time_days"]
df_daily[snapshot_columns] = df_daily.groupby("product_name")[snapshot_columns].ffill()

df_daily["quantity_sold"] = df_daily["quantity_sold"].astype(int)

print("Complete grid rows:", len(df_daily))
print("Added zero-sales product-days:", int((df_daily["quantity_sold"] == 0).sum()))
df_daily.head()

Complete grid rows: 6168
Added zero-sales product-days: 2707


,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,supplier_lead_time_days,product_category,weather_condition
0,Bag Delivery 45L,2025-01-01,5,889.97,414.97,7.5,178.1725,101.0,3.0,Bag Entrega,Ensolarado
1,Bag Delivery 45L,2025-01-02,2,364.80,174.80,0.0,182.4000,53.0,7.0,Bag Entrega,Ensolarado
2,Bag Delivery 45L,2025-01-03,1,183.92,88.92,5.0,183.9200,24.0,7.0,Bag Entrega,Ensolarado
3,Bag Delivery 45L,2025-01-04,3,536.25,251.25,5.0,178.7500,52.0,7.0,Bag Entrega,Ensolarado
4,Bag Delivery 45L,2025-01-05,8,1442.82,682.82,4.0,179.0000,109.0,8.0,Bag Entrega,Ensolarado


## Validate Final Dataset

In [6]:
expected_rows = df_daily["product_name"].nunique() * df_daily["sale_date"].nunique()
key_columns = ["product_name", "sale_date", "quantity_sold", "product_category", "weather_condition"]

assert len(df_daily) == expected_rows
assert not df_daily.duplicated(["product_name", "sale_date"]).any()
assert df_daily[key_columns].notna().all().all()
assert (df_daily["quantity_sold"] >= 0).all()
assert (df_daily["discount_pct"].between(0, 100)).all()

validation_df = pd.DataFrame({
    "check": [
        "Rows in complete grid",
        "Duplicate product-dates",
        "Missing key values",
        "Negative quantity values",
        "Missing initial business snapshots",
    ],
    "result": [
        len(df_daily),
        int(df_daily.duplicated(["product_name", "sale_date"]).sum()),
        int(df_daily[key_columns].isna().sum().sum()),
        int((df_daily["quantity_sold"] < 0).sum()),
        int(df_daily[snapshot_columns].isna().sum().sum()),
    ],
})
validation_df

,check,result
0,Rows in complete grid,6168
1,Duplicate product-dates,0
2,Missing key values,0
3,Negative quantity values,0
4,Missing initial business snapshots,72


## Save Processed Dataset

In [7]:
OUTPUT_PATH = ROOT / "data" / "processed" / "daily_product_sales.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_daily.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print("Saved:", OUTPUT_PATH.resolve())
print("Shape:", df_daily.shape)

Saved:

 C:\DEV\motostock-ai\data\processed\daily_product_sales.csv
Shape: (6168, 11)


## Summary

The transaction data is now a complete daily product dataset. Every product has one row for every date. Days without transactions have zero demand, while stock and other snapshot values are carried forward from the last known observation.

No model is trained here. The next notebook will use this daily dataset to create simple forecasting baselines.